## MedDec Decision Extraction, LLM Based | Colab Notebook

**Goal**: Run the zero-shot and one-shot promps through LLM pipelines to detect medical decision spans. Experiments with a seq2seq model `FLAN-T5` and a causal language model `Llama-3-8B`

## Step 0. Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
# sentencepiece (FLAN-T5 tokeniser)
# accelerate is required for device_map='auto' (Llama-3)
!pip install -q transformers sentencepiece accelerate

In [ ]:
!pip install python-dotenv

In [3]:
import shutil, sys
from pathlib import Path
from dotenv import load_dotenv

# EDIT PATHS
DRIVE_CODE_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/04 Code/04 Code/med-decision-extraction")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/02 Data")
ENV_DIR = Path("/content/drive/MyDrive/AI4H-project-rework/04 Code/04 Code/.env")

LOCAL_CODE = Path("/content/meddec")
if LOCAL_CODE.exists():
    shutil.rmtree(LOCAL_CODE)
shutil.copytree(DRIVE_CODE_DIR, LOCAL_CODE)
sys.path.insert(0, str(LOCAL_CODE))

MEDDEC_DIR  = DRIVE_DATA_DIR / "meddec-mimic-iii"
SPLITS_DIR  = MEDDEC_DIR / "splits"
GENS_DIR    = DRIVE_DATA_DIR / "gens"   # predictions saved here (persisted to Drive)
GENS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_DIR)

print("Code :", LOCAL_CODE)
print("Data :", MEDDEC_DIR)

Code : /content/meddec
Data : /content/drive/MyDrive/AI4H-project-rework/02 Data/meddec-mimic-iii


## Step 0b. Verify GPU + data, pick LLM model

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

n_json = len(list((MEDDEC_DIR / "data").glob("*.json")))
n_txt  = len(list((MEDDEC_DIR / "raw_text").glob("*.txt")))
print(f"\nJSON annotations : {n_json}")
print(f"Raw text files   : {n_txt}")
print(f"Test split size  : {len((SPLITS_DIR / 'test.txt').read_text().splitlines())} notes")

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB

JSON annotations : 403
Raw text files   : 403
Test split size  : 41 notes


In [ ]:
# Pick seq2seq FLAN-T5 model
MODEL_NAME = "google/flan-t5-xl"

#"google/flan-t5-small"   # ~0.3 GB  — fast test, weakest quality
#"google/flan-t5-base"  # ~1 GB    — better quality, still fast on T4
#"google/flan-t5-xl"    # ~12 GB   — best free-tier quality

In [ ]:
# Set Llama-3 model (TBD)
import os
hf_token = os.getenv("HF_TOKEN")
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"  # ~16 GB might not work on Colab T4 free tier
# For Llama-3, set your HuggingFace token (request access at huggingface.co first)
# import os; os.environ["HF_TOKEN"] = "hf_..."

print(f"Selected model: {MODEL_NAME}")

## Seq2Seq generative model (FLAN-T5)

### Step 1. Zero-shot generation

For each note and for each of the 9 categories, give the zero shot prompt to the LLM with the noteattached

In [5]:
# Save output of detected spans for each note here
ZERO_SHOT_DIR = GENS_DIR / "test_zero_shot"

In [ ]:
from gen_span_detection import run_pipeline

run_pipeline(meddec_dir = MEDDEC_DIR,
    splits_dir = SPLITS_DIR,
    model_name = MODEL_NAME,
    output_dir = ZERO_SHOT_DIR,
    split      = "test",
    mode       = "zero_shot",
    #max_samples = 5,   # uncomment to process only x notes for testing
)

Loading google/flan-t5-xl on cuda ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  Model type : seq2seq
  Parameters : 2850M




zero_shot | test:   0%|          | 0/41 [00:00<?, ?it/s]

zero_shot | test:   2%|▏         | 1/41 [00:31<20:43, 31.08s/it]

zero_shot | test:   5%|▍         | 2/41 [01:01<19:46, 30.42s/it]

zero_shot | test:   7%|▋         | 3/41 [01:34<20:07, 31.79s/it]

zero_shot | test:  10%|▉         | 4/41 [01:58<17:36, 28.56s/it]

zero_shot | test:  12%|█▏        | 5/41 [02:38<19:44, 32.91s/it]

zero_shot | test:  15%|█▍        | 6/41 [03:14<19:47, 33.93s/it]

zero_shot | test:  17%|█▋        | 7/41 [03:56<20:43, 36.57s/it]

zero_shot | test:  20%|█▉        | 8/41 [04:16<17:12, 31.30s/it]

zero_shot | test:  22%|██▏       | 9/41 [04:58<18:25, 34.56s/it]

zero_shot | test:  24%|██▍       | 10/41 [05:25<16:43, 32.38s/it]

zero_shot | test:  27%|██▋       | 11/41 [05:49<14:52, 29.75s/it]

zero_shot | test:  29%|██▉       | 12/41 [06:22<14:52, 30.78s/it]

zero_shot | test:  32%|███▏      | 13/41 [06:39<12:23, 26.54s/it]

zero_shot | test:  34%|███▍      | 14/41 [07:03<11:35, 25.76s/it]

zero_shot |


Done. Skipped 0 files. Results in /content/drive/MyDrive/AI4H-project-rework/02 Data/gens/test_zero_shot


### Step 2. One-shot generation

Same pipeline, but each prompt now includes one example drawn from the **same note's annotations**.  
(note: The example's category is whichever category has the most spans in that note and is always different from the target category).

In [6]:
# Save output of detected spans for each note here
ONE_SHOT_DIR = GENS_DIR / "test_one_shot"

In [ ]:
from gen_span_detection import run_pipeline

run_pipeline(
    meddec_dir = MEDDEC_DIR,
    splits_dir = SPLITS_DIR,
    model_name = MODEL_NAME,
    output_dir = ONE_SHOT_DIR,
    split      = "test",
    mode       = "one_shot",
    # max_samples = 5,
)

Loading google/flan-t5-xl on cuda ...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  Model type : seq2seq
  Parameters : 2850M




one_shot | test:   0%|          | 0/41 [00:00<?, ?it/s]

one_shot | test:   2%|▏         | 1/41 [00:18<12:01, 18.04s/it]

one_shot | test:   5%|▍         | 2/41 [01:16<27:21, 42.08s/it]

one_shot | test:   7%|▋         | 3/41 [01:27<17:33, 27.72s/it]

one_shot | test:  10%|▉         | 4/41 [02:21<23:27, 38.05s/it]

one_shot | test:  12%|█▏        | 5/41 [03:28<29:10, 48.61s/it]

one_shot | test:  15%|█▍        | 6/41 [04:11<27:05, 46.43s/it]

one_shot | test:  17%|█▋        | 7/41 [04:48<24:35, 43.40s/it]

one_shot | test:  20%|█▉        | 8/41 [05:49<26:58, 49.05s/it]

one_shot | test:  22%|██▏       | 9/41 [06:43<27:04, 50.77s/it]

one_shot | test:  24%|██▍       | 10/41 [07:16<23:23, 45.27s/it]

one_shot | test:  27%|██▋       | 11/41 [07:26<17:14, 34.49s/it]

one_shot | test:  29%|██▉       | 12/41 [07:35<12:49, 26.55s/it]

one_shot | test:  32%|███▏      | 13/41 [07:48<10:31, 22.54s/it]

one_shot | test:  34%|███▍      | 14/41 [07:56<08:11, 18.19s/it]

one_shot | test:  37%|███▋


Done. Skipped 0 files. Results in /content/drive/MyDrive/AI4H-project-rework/02 Data/gens/test_one_shot


### Step 3. Evaluation
Methods:
- **em** (exact match): `pred.strip() == gold.strip()`
- **approx-m**: one is a substring of the other AND word count difference ≤ 10

In [7]:
from eval_gen import evaluate_predictions, print_results

TEST_SPLIT_FILE = SPLITS_DIR / "test.txt"

results_zero_em     = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ZERO_SHOT_DIR, method="em")
results_zero_approx = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ZERO_SHOT_DIR, method="approx-m")
results_one_em      = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ONE_SHOT_DIR,  method="em")
results_one_approx  = evaluate_predictions(MEDDEC_DIR, TEST_SPLIT_FILE, ONE_SHOT_DIR,  method="approx-m")

Gold triples : 5188
Pred triples : 284
Gold triples : 5188
Pred triples : 284
Gold triples : 5188
Pred triples : 274
Gold triples : 5188
Pred triples : 274


In [8]:
print_results(results_zero_em,     label="Zero-shot | EM")
print_results(results_zero_approx, label="Zero-shot | Approx-M")
print_results(results_one_em,      label="One-shot  | EM")
print_results(results_one_approx,  label="One-shot  | Approx-M")


  LLM Span F1 (em) — Zero-shot | EM
  Gold triples :  5188
  Pred triples :   284
  TP=0  FP=284  FN=5188
  Precision  : 0.0000
  Recall     : 0.0000
  F1         : 0.0000  ← main metric

  Cat  Name                           F1       P       R   Gold   Pred
  --------------------------------------------------------------
    1  Contact-related             0.000   0.000   0.000    269     32
    2  Gathering information       0.000   0.000   0.000     39     30
    3  Defining problem            0.000   0.000   0.000   2030     22
    4  Treatment goal              0.000   0.000   0.000     14     36
    5  Drug                        0.000   0.000   0.000   1334     36
    6  Therapeutic procedure       0.000   0.000   0.000    527     34
    7  Evaluating test result      0.000   0.000   0.000    744     29
    8  Deferment                   0.000   0.000   0.000      8     33
    9  Advice and precaution       0.000   0.000   0.000    223     32


  LLM Span F1 (approx-m) — Zero-sh